# Lesson 7: Numerosity estimation with flexible encoding functions

This lesson introduces **Bayesian observer models for numerosity estimation**, informed by:
- *"Distributed range adaptation"* (de Hollander et al.) — neural tuning curves remap with prior width
- *"Endogenous precision of the number sense"* (Prat-Carrabin & Woodford) — resource-rational encoding with motor noise

## The task

Subjects see a cloud of dots and estimate the number. Two blocked conditions share the **same encoding** but have **different prior ranges**:
- **Narrow**: n ~ Uniform[10, 25]
- **Wide**: n ~ Uniform[10, 40]

## The model

The generative process has three stages:
1. **Encoding** (shared): n → r ~ N(μ(n), ν²), where μ(n) is a smooth monotone function
2. **Bayesian decoding** (condition-specific): n̂(r) = E[n | r] under Uniform[n_min, n_max]
3. **Motor noise** (shared): response ~ N(n̂(r), σ_motor²)

The key constraint: encoding μ(n) and noise ν are **shared across conditions**. Only the decoding prior changes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from bauer.utils.data import load_neuralpriors
from bauer.numerosity import LogEncodingEstimationModel, FlexibleEncodingEstimationModel

## 1. Load neural_priors data

The data is bundled in the bauer package (39 subjects, ~480 trials each).

In [ ]:
df = load_neuralpriors()
print(f'Loaded {len(df)} trials from {df.index.get_level_values("subject").nunique()} subjects')
print(f'Conditions: {df["range"].unique().tolist()}')

# Plot one subject's data — both conditions
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True, sharex=True)
sub1 = df.xs(1, level='subject')
for ax, cond in zip(axes, ['narrow', 'wide']):
    d = sub1[sub1['range'] == cond]
    ax.scatter(d['n'], d['response'], alpha=0.3, s=10)
    n_range = [10, 25] if cond == 'narrow' else [10, 40]
    ax.plot(n_range, n_range, 'r--', label='Identity line')
    ax.set_xlabel('True numerosity')
    ax.set_ylabel('Response')
    ax.set_title(f'Subject 1 — {cond} [10, {n_range[1]}]')
    ax.legend()
plt.tight_layout()
plt.show()

## 2. Fit the Log-encoding model

The simplest model: μ(n) = log(n) (fixed), with shared ν and σ_motor across conditions. Only the Bayesian decoding prior differs (Uniform[10,25] vs Uniform[10,40]).

In [ ]:
# Fit one subject — BOTH narrow and wide conditions jointly
sub1 = df.xs(1, level='subject').reset_index()
paradigm = sub1[['n', 'response', 'range']].dropna(subset=['response']).copy()
paradigm['n'] = paradigm['n'].astype(float)
paradigm.index = pd.MultiIndex.from_arrays(
    [np.ones(len(paradigm), dtype=int), range(len(paradigm))],
    names=['subject', 'trial'])

print(f'Trials: {len(paradigm)} ({(paradigm["range"]=="narrow").sum()} narrow, '
      f'{(paradigm["range"]=="wide").sum()} wide)')

model = LogEncodingEstimationModel(paradigm, grid_resolution=51, n_min=10, n_max=40,
                                    response_bin_width=1.0)
model.build_estimation_model(paradigm, hierarchical=False, flat_prior=True)

# Check gradients work
with model.estimation_model:
    dlogp = model.estimation_model.compile_dlogp()
    print(f'Gradients: {dlogp(model.estimation_model.initial_point())}')

pars = model.fit_map(progressbar=False)
print(f'\nFitted parameters:')
print(f'  nu = {float(pars["nu"]):.4f} (sensory noise in log-number space)')
print(f'  sigma_motor = {float(pars["sigma_motor"]):.2f} (motor noise in number space)')

## 2b. Posterior predictive checks

Now let's check how well the fitted model captures the data. We use `predict()` to get the expected response per trial, and `simulate()` to draw synthetic data from the model.

In [ ]:
# Predict mean and SD for each trial
predictions = model.predict(paradigm, pars)

# Simulate 10 synthetic datasets
simulations = model.simulate(paradigm, pars, n_samples=10)
print(f'Predictions shape: {predictions.shape}')
print(f'Simulations shape: {simulations.shape}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for col, cond in enumerate(['narrow', 'wide']):
    mask = predictions['range'] == cond
    pred_cond = predictions[mask]
    data_cond = paradigm[paradigm['range'] == cond]
    n_range = [10, 25] if cond == 'narrow' else [10, 40]

    # --- Top row: mean response vs true numerosity ---
    ax = axes[0, col]
    # Empirical mean per stimulus
    emp_mean = data_cond.groupby('n')['response'].mean()
    emp_se = data_cond.groupby('n')['response'].sem()
    ax.errorbar(emp_mean.index, emp_mean.values, yerr=emp_se.values * 1.96,
                fmt='o', color='C0', label='Data (mean ± 95% CI)', zorder=3)

    # Model predicted mean per stimulus
    pred_mean = pred_cond.groupby('n')['predicted_mean'].mean()
    ax.plot(pred_mean.index, pred_mean.values, 's-', color='C1',
            label='Model prediction', markersize=5)

    ax.plot(n_range, n_range, 'k--', alpha=0.3, label='Identity')
    ax.set_xlabel('True numerosity')
    ax.set_ylabel('Mean response')
    ax.set_title(f'{cond.capitalize()} [{n_range[0]}, {n_range[1]}]')
    ax.legend(fontsize=8)

    # --- Bottom row: response SD vs true numerosity ---
    ax = axes[1, col]
    emp_sd = data_cond.groupby('n')['response'].std()
    ax.plot(emp_sd.index, emp_sd.values, 'o', color='C0', label='Data SD')

    pred_sd = pred_cond.groupby('n')['predicted_sd'].mean()
    ax.plot(pred_sd.index, pred_sd.values, 's-', color='C1',
            label='Model predicted SD', markersize=5)

    ax.set_xlabel('True numerosity')
    ax.set_ylabel('Response SD')
    ax.set_title(f'{cond.capitalize()} — variability')
    ax.legend(fontsize=8)

plt.suptitle(f'Subject 1 — Log-encoding model (ν={float(pars["nu"]):.3f}, '
             f'σ_motor={float(pars["sigma_motor"]):.2f})', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Simulated data vs real data
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True, sharex=True)

for ax, cond in zip(axes, ['narrow', 'wide']):
    data_cond = paradigm[paradigm['range'] == cond]
    sim_cond = simulations[simulations['range'] == cond]
    n_range = [10, 25] if cond == 'narrow' else [10, 40]

    # Plot simulated data (one sample) as background
    sim_sample1 = sim_cond.xs(1, level='sample')
    ax.scatter(sim_sample1['n'], sim_sample1['simulated_response'],
               alpha=0.15, s=8, color='C1', label='Simulated', zorder=1)

    # Plot real data on top
    ax.scatter(data_cond['n'], data_cond['response'],
               alpha=0.4, s=12, color='C0', label='Data', zorder=2)

    ax.plot(n_range, n_range, 'k--', alpha=0.3)
    ax.set_xlabel('True numerosity')
    ax.set_ylabel('Response')
    ax.set_title(f'{cond.capitalize()} [{n_range[0]}, {n_range[1]}]')
    ax.legend(fontsize=9)

plt.suptitle('Real data vs simulated data from fitted model', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Fit all subjects

Fit individual MLE per subject, with shared noise parameters across narrow and wide conditions.

In [ ]:
# Prepare multi-subject data
all_paradigms = []
for sub_id in df.index.get_level_values('subject').unique()[:5]:  # first 5 subjects for speed
    sub_data = df.xs(sub_id, level='subject').reset_index()
    p = sub_data[['n', 'response', 'range']].dropna(subset=['response']).copy()
    p['n'] = p['n'].astype(float)
    p['subject'] = sub_id
    all_paradigms.append(p)

paradigm_all = pd.concat(all_paradigms, ignore_index=True)
paradigm_all = paradigm_all.set_index(['subject', paradigm_all.groupby('subject').cumcount()])
paradigm_all.index.names = ['subject', 'trial']

model = LogEncodingEstimationModel(paradigm_all, grid_resolution=51, n_min=10, n_max=40,
                                    response_bin_width=1.0)
mle_results = model.fit_map_individual(paradigm_all, flat_prior=True)
print('Fitted parameters per subject:')
print(mle_results.round(4))

In [ ]:
# PPC for all fitted subjects: predicted bias (mean response - true n) vs true n
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for ax, cond in zip(axes, ['narrow', 'wide']):
    n_range = [10, 25] if cond == 'narrow' else [10, 40]

    for sub_id in mle_results.index:
        sub_data = paradigm_all.xs(sub_id, level='subject')
        sub_pars = mle_results.loc[[sub_id]].to_dict(orient='records')[0]

        sub_model = LogEncodingEstimationModel(sub_data, grid_resolution=51,
                                                n_min=10, n_max=40, response_bin_width=1.0)
        pred = sub_model.predict(sub_data, sub_pars)

        mask = pred['range'] == cond
        if mask.sum() == 0:
            continue

        pred_cond = pred[mask]
        # Predicted bias = predicted_mean - true n
        pred_bias = pred_cond.groupby('n').apply(
            lambda g: (g['predicted_mean'] - g['n']).mean())
        ax.plot(pred_bias.index, pred_bias.values, 'o-', color='C1', alpha=0.4, markersize=3)

        # Empirical bias
        data_cond = sub_data[sub_data['range'] == cond]
        emp_bias = data_cond.groupby('n').apply(
            lambda g: (g['response'] - g['n']).mean())
        ax.plot(emp_bias.index, emp_bias.values, 's-', color='C0', alpha=0.4, markersize=3)

    ax.axhline(0, color='k', ls='--', alpha=0.3)
    ax.set_xlabel('True numerosity')
    ax.set_ylabel('Bias (response - true)')
    ax.set_title(f'{cond.capitalize()} [{n_range[0]}, {n_range[1]}]')

# Custom legend
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], color='C0', marker='s', label='Data'),
                   Line2D([0], [0], color='C1', marker='o', label='Model')]
axes[0].legend(handles=legend_elements, fontsize=9)
plt.suptitle('Estimation bias: data vs model predictions (5 subjects)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Flexible encoding with condition-specific mappings

Instead of fixing μ(n) = log(n), we let it be a **monotone Bernstein polynomial**. Crucially, we fit **separate encoding functions per condition** — the narrow and wide ranges may induce different internal representations (distributed range adaptation). Noise parameters (ν, σ_motor) remain shared.

This is the key model: same noise, different encoding per condition.

In [ ]:
# Fit flexible encoding with condition-specific mappings
sub1 = df.xs(1, level='subject').reset_index()
paradigm = sub1[['n', 'response', 'range']].dropna(subset=['response']).copy()
paradigm['n'] = paradigm['n'].astype(float)
paradigm.index = pd.MultiIndex.from_arrays(
    [np.ones(len(paradigm), dtype=int), range(len(paradigm))],
    names=['subject', 'trial'])

# Condition-specific encoding (default)
flex_model = FlexibleEncodingEstimationModel(
    paradigm, grid_resolution=51, n_poly=6, n_min=10, n_max=40,
    response_bin_width=1.0, condition_specific_encoding=True)
flex_model.build_estimation_model(paradigm, hierarchical=False, flat_prior=True)
flex_pars = flex_model.fit_map(progressbar=False)

print(f'nu = {float(flex_pars["nu"]):.4f} (shared)')
print(f'sigma_motor = {float(flex_pars["sigma_motor"]):.2f} (shared)')

# Plot the encoding functions
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# --- Left: encoding functions per condition ---
ax = axes[0]
n_ref = np.linspace(10, 40, 200)
log_ref = np.log(n_ref)
log_ref_norm = (log_ref - log_ref[0]) / (log_ref[-1] - log_ref[0])
ax.plot(n_ref, log_ref_norm, 'k--', alpha=0.5, lw=1.5, label='log(n)')

for cond, color, ls in [('narrow', 'C0', '-'), ('wide', 'C1', '-')]:
    n_grid, mu_grid = flex_model.get_encoding_function(flex_pars, condition=cond)
    mu_norm = (mu_grid - mu_grid.min()) / (mu_grid.max() - mu_grid.min() + 1e-10)
    ax.plot(n_grid, mu_norm, ls, color=color, lw=2.5, label=f'{cond.capitalize()}')
    # Mark the condition's prior range
    n_max_c = 25 if cond == 'narrow' else 40
    ax.axvline(n_max_c, color=color, ls=':', alpha=0.4)

ax.set_xlabel('Numerosity')
ax.set_ylabel('μ(n) (normalized)')
ax.set_title('Encoding functions')
ax.legend()

# --- Middle & Right: predictions per condition ---
predictions = flex_model.predict(paradigm, flex_pars)

for col, cond in enumerate(['narrow', 'wide']):
    ax = axes[col + 1]
    data_cond = paradigm[paradigm['range'] == cond]
    pred_cond = predictions[predictions['range'] == cond]
    n_range = [10, 25] if cond == 'narrow' else [10, 40]

    emp_mean = data_cond.groupby('n')['response'].mean()
    emp_se = data_cond.groupby('n')['response'].sem()
    ax.errorbar(emp_mean.index, emp_mean.values, yerr=emp_se.values * 1.96,
                fmt='o', color='C0' if cond == 'narrow' else 'C1',
                label='Data (mean ± 95% CI)', zorder=3)

    pred_mean = pred_cond.groupby('n')['predicted_mean'].mean()
    ax.plot(pred_mean.index, pred_mean.values, 's-', color='C2',
            label='Model prediction', markersize=4)

    ax.plot(n_range, n_range, 'k--', alpha=0.3)
    ax.set_xlabel('True numerosity')
    ax.set_ylabel('Mean response')
    ax.set_title(f'{cond.capitalize()} [{n_range[0]}, {n_range[1]}]')
    ax.legend(fontsize=8)

plt.suptitle(f'Subject 1 — Condition-specific flexible encoding '
             f'(ν={float(flex_pars["nu"]):.3f}, σ_motor={float(flex_pars["sigma_motor"]):.2f})',
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Compare Log vs Flexible encoding predictions (using pooled-session models)
# Re-fit flexible model on all trials for comparison
paradigm = sub1[['n', 'response', 'range']].dropna(subset=['response']).copy()
paradigm['n'] = paradigm['n'].astype(float)
paradigm.index = pd.MultiIndex.from_arrays(
    [np.ones(len(paradigm), dtype=int), range(len(paradigm))],
    names=['subject', 'trial'])

flex_model_pooled = FlexibleEncodingEstimationModel(paradigm, grid_resolution=51,
                                                     n_poly=6, n_min=10, n_max=40,
                                                     response_bin_width=1.0)
flex_model_pooled.build_estimation_model(paradigm, hierarchical=False, flat_prior=True)
flex_pars_pooled = flex_model_pooled.fit_map(progressbar=False)

log_pred = model.predict(paradigm, pars)
flex_pred = flex_model_pooled.predict(paradigm, flex_pars_pooled)

# Show the encoding function
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
n_grid, mu_grid = flex_model_pooled.get_encoding_function(flex_pars_pooled)
mu_norm = (mu_grid - mu_grid[0]) / (mu_grid[-1] - mu_grid[0] + 1e-10)
ax.plot(n_grid, mu_norm, '-', color='C2', lw=2, label='Flexible (fitted)')
log_ref = np.log(n_grid)
log_ref_norm = (log_ref - log_ref[0]) / (log_ref[-1] - log_ref[0])
ax.plot(n_grid, log_ref_norm, 'k--', alpha=0.5, label='log(n)')
lin_ref = (n_grid - n_grid[0]) / (n_grid[-1] - n_grid[0])
ax.plot(n_grid, lin_ref, ':', color='gray', alpha=0.5, label='linear')
ax.set_xlabel('Numerosity')
ax.set_ylabel('μ(n) (normalized)')
ax.set_title('Encoding function')
ax.legend()

for col, cond in enumerate(['narrow', 'wide']):
    ax = axes[col + 1]
    data_cond = paradigm[paradigm['range'] == cond]
    n_range = [10, 25] if cond == 'narrow' else [10, 40]

    emp_mean = data_cond.groupby('n')['response'].mean()
    emp_se = data_cond.groupby('n')['response'].sem()
    ax.errorbar(emp_mean.index, emp_mean.values, yerr=emp_se.values * 1.96,
                fmt='o', color='C0', label='Data', zorder=3)

    for pred_df, name, color in [(log_pred, 'Log encoding', 'C1'),
                                  (flex_pred, 'Flexible', 'C2')]:
        m = pred_df[pred_df['range'] == cond].groupby('n')['predicted_mean'].mean()
        ax.plot(m.index, m.values, '-', color=color, label=name)

    ax.plot(n_range, n_range, 'k--', alpha=0.3)
    ax.set_xlabel('True numerosity')
    ax.set_ylabel('Mean response')
    ax.set_title(f'{cond.capitalize()}')
    ax.legend(fontsize=8)

plt.suptitle('Log vs flexible encoding — Subject 1', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Condition-specific efficient coding

The `EfficientEncodingEstimationModel` adapts the encoding function to each condition's prior. Under efficient coding, the CDF of the prior serves as the encoding transform. For a uniform prior on [n_min, n_max] with log encoding:

$$F_c(n) = \frac{\log n - \log n_{\min,c}}{\log n_{\max,c} - \log n_{\min,c}}$$

This means the narrow and wide conditions use different encoding functions, but share the same noise level ν in representation space.

In [ ]:
from bauer.numerosity import EfficientEncodingEstimationModel

# Fit to subject 1
eff_model = EfficientEncodingEstimationModel(paradigm, grid_resolution=51,
                                              n_min=10, n_max=40, response_bin_width=1.0)
eff_model.build_estimation_model(paradigm, hierarchical=False, flat_prior=True)
eff_pars = eff_model.fit_map(progressbar=False)
eff_pred = eff_model.predict(paradigm, eff_pars)

print(f'Efficient encoding: nu={float(eff_pars["nu"]):.4f}, sigma_motor={float(eff_pars["sigma_motor"]):.2f}')

# Compare all three models
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

models_preds = [
    ('Log encoding', log_pred, 'C1'),
    ('Flexible encoding', flex_pred, 'C2'),
    ('Efficient encoding', eff_pred, 'C3'),
]

for col, cond in enumerate(['narrow', 'wide']):
    data_cond = paradigm[paradigm['range'] == cond]
    n_range = [10, 25] if cond == 'narrow' else [10, 40]

    # Mean response
    ax = axes[0, col]
    emp_mean = data_cond.groupby('n')['response'].mean()
    emp_se = data_cond.groupby('n')['response'].sem()
    ax.errorbar(emp_mean.index, emp_mean.values, yerr=emp_se.values * 1.96,
                fmt='o', color='C0', label='Data', zorder=3)
    for name, pred, color in models_preds:
        m = pred[pred['range'] == cond].groupby('n')['predicted_mean'].mean()
        ax.plot(m.index, m.values, '-', color=color, label=name, markersize=3)
    ax.plot(n_range, n_range, 'k--', alpha=0.3)
    ax.set_xlabel('True numerosity')
    ax.set_ylabel('Mean response')
    ax.set_title(f'{cond.capitalize()} — mean')
    ax.legend(fontsize=7)

    # Bias
    ax = axes[1, col]
    emp_bias = data_cond.groupby('n').apply(lambda g: (g['response'] - g['n']).mean())
    ax.plot(emp_bias.index, emp_bias.values, 'o', color='C0', label='Data')
    for name, pred, color in models_preds:
        p = pred[pred['range'] == cond]
        m_bias = p.groupby('n').apply(lambda g: (g['predicted_mean'] - g['n']).mean())
        ax.plot(m_bias.index, m_bias.values, '-', color=color, label=name)
    ax.axhline(0, color='k', ls='--', alpha=0.3)
    ax.set_xlabel('True numerosity')
    ax.set_ylabel('Bias (response - true)')
    ax.set_title(f'{cond.capitalize()} — bias')
    ax.legend(fontsize=7)

plt.suptitle('Model comparison: mean response and bias', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Encoding functions across subjects

Do different subjects have different internal number lines? Let's fit the flexible encoding model to several subjects and overlay the recovered encoding functions.

In [ ]:
# Fit flexible encoding to first 10 subjects
subject_ids = df.index.get_level_values('subject').unique()[:10]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: encoding functions
ax = axes[0]
n_ref = np.linspace(10, 40, 200)
log_ref = np.log(n_ref)
log_ref_norm = (log_ref - log_ref[0]) / (log_ref[-1] - log_ref[0])
ax.plot(n_ref, log_ref_norm, 'k--', lw=2, alpha=0.7, label='log(n)')

all_flex_pars = {}
for sub_id in subject_ids:
    sub_data = df.xs(sub_id, level='subject').reset_index()
    p = sub_data[['n', 'response', 'range']].dropna(subset=['response']).copy()
    p['n'] = p['n'].astype(float)
    p.index = pd.MultiIndex.from_arrays(
        [np.full(len(p), sub_id), range(len(p))],
        names=['subject', 'trial'])

    fm = FlexibleEncodingEstimationModel(p, grid_resolution=51, n_poly=6,
                                          n_min=10, n_max=40, response_bin_width=1.0)
    fm.build_estimation_model(p, hierarchical=False, flat_prior=True)
    fp = fm.fit_map(progressbar=False)
    all_flex_pars[sub_id] = fp

    n_grid, mu_grid = fm.get_encoding_function(fp)
    mu_norm = (mu_grid - mu_grid[0]) / (mu_grid[-1] - mu_grid[0] + 1e-10)
    ax.plot(n_grid, mu_norm, '-', alpha=0.5, lw=1.5)

ax.set_xlabel('Numerosity')
ax.set_ylabel('μ(n) (normalized)')
ax.set_title('Fitted encoding functions (10 subjects)')
ax.legend()

# Right: nu vs sigma_motor
ax = axes[1]
nus = [float(all_flex_pars[s]['nu']) for s in subject_ids]
sigmas = [float(all_flex_pars[s]['sigma_motor']) for s in subject_ids]
ax.scatter(nus, sigmas, s=40, zorder=3)
for i, s in enumerate(subject_ids):
    ax.annotate(str(s), (nus[i], sigmas[i]), fontsize=7, ha='left')
ax.set_xlabel('ν (sensory noise)')
ax.set_ylabel('σ_motor (motor noise)')
ax.set_title('Noise parameters across subjects')

plt.tight_layout()
plt.show()